# Proyecto Integrador Módulo 4
## Redes Neuronales Profundas: de Densa a Convolucional

---
# Parte 1: Clasificador CNN para MNIST


**Instrucciones para ti (bórralas antes de entregar):**
Este código fue generado con ayuda de IA y funciona, pero los **comentarios que
expliquen qué hace y por qué** los tienes que escribir tú, con tus propias palabras,
en las celdas de markdown marcadas como `TU EXPLICACIÓN AQUÍ`. No es válido usar
IA para esa parte. Si hay algo que no entiendes del todo, dilo explícitamente.

## 1. Librerías

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)


> **TU EXPLICACIÓN AQUÍ:** ¿qué hace este bloque y por qué se necesita cada librería?

## 2. Descarga y preparación del dataset MNIST

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Ejemplos de entrenamiento: {len(train_dataset)}")
print(f"Ejemplos de prueba: {len(test_dataset)}")

# Vistazo rápido a algunas imágenes
fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for i, ax in enumerate(axes):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()


> **TU EXPLICACIÓN AQUÍ:** ¿qué hace `transforms.Normalize`? ¿por qué separamos train/test? ¿qué es el `DataLoader` y para qué sirve `batch_size`?

## 3. Arquitectura de la red CNN

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))   # 28x28 -> 14x14
        x = self.pool(self.relu(self.conv2(x)))   # 14x14 -> 7x7
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = CNN().to(device)
print(model)


> **TU EXPLICACIÓN AQUÍ:** explica capa por capa qué hace `conv1`, `pool`, `conv2` y las capas `fc`. ¿Por qué el tamaño pasa de 28x28 a 7x7? ¿Qué papel juega `ReLU`?

## 4. Entrenamiento del modelo

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 5
train_losses = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    train_losses.append(epoch_loss)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f}")


> **TU EXPLICACIÓN AQUÍ:** ¿qué es `criterion`? ¿qué hace `optimizer.zero_grad()`, `loss.backward()` y `optimizer.step()`? ¿por qué se repite esto por cada `epoch`?

## 5. Evolución del error durante el entrenamiento

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, num_epochs + 1), train_losses, marker="o")
plt.xlabel("Época")
plt.ylabel("Pérdida (loss)")
plt.title("Evolución del error durante el entrenamiento")
plt.grid(True)
plt.show()


> **TU EXPLICACIÓN AQUÍ:** ¿qué significa que la curva baje (o no baje) a lo largo de las épocas?

## 6. Evaluación en el conjunto de prueba

In [ ]:
model.eval()
all_preds = []
all_labels = []
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = 100 * correct / total
print(f"Accuracy en el conjunto de prueba: {accuracy:.2f}%")


> **TU EXPLICACIÓN AQUÍ:** ¿por qué se usa `model.eval()` y `torch.no_grad()` aquí y no durante el entrenamiento?

## 7. Matriz de confusión

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel("Predicción")
plt.ylabel("Etiqueta real")
plt.title("Matriz de confusión - CNN sobre MNIST")
plt.show()


> **TU EXPLICACIÓN AQUÍ:** ¿qué dígitos confunde más el modelo? ¿tienes una hipótesis de por qué? (esto también lo vas a retomar en la Parte 2)

---
### Notas para ti antes de seguir
- Revisa que todas las celdas `TU EXPLICACIÓN AQUÍ` estén completas con tus propias palabras antes de pasar a la Parte 2.
- Si algo del código no te quedó claro del todo, agrégalo aquí explícitamente — la consigna lo pide.
- Una vez lista la Parte 1, continúa con la Parte 2 (comparación con la red densa) y la Parte 3 (reflexión personal) en este mismo notebook.
